In [0]:
"""
Bonus: Multi-Head delay modeling configuration.

This file defines a multi-head model using:
    - The best-performing NN and GBT settings from prior iterations
      (strong OTPA and RMSE).
    - A "balanced" severe-delay head with *no* class weighting,
      so SDDR is plain (unweighted) recall at a moderate threshold.

Heads:
    1) Binary NN classifier for OTPA (DEP_DELAY < 15 vs >= 15)
    2) GBTRegressor for continuous RMSE (DEP_DELAY)
    3) LogisticRegression for severe delays (SEVERE_DEL60) with
       unweighted recall/precision, threshold ~= 0.3

The cross-validator reports, per fold:
    - nn_otpa
    - reg_rmse
    - reg_mae
    - severe_sddr
    - severe_sddr_prec
"""

import importlib.util

from pyspark.sql import functions as F
from pyspark.sql.functions import col, isnan, regexp_replace, when, length, trim
from pyspark.sql.types import DoubleType, StringType

from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier, LogisticRegression
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import pandas as pd


# -------------------------------------------------------------------------
# Config: head behavior
# -------------------------------------------------------------------------

# NN OTPA head: keep architecture [input, 32, 16, 2] with 50 iterations,
# which performed best in prior 5-year runs.
NN_MAX_ITER = 50

# Severe-delay head: probability threshold for classifying a flight as
# "severe delay" in the evaluator. Using a moderate value here (~0.3)
# gave a more balanced SDDR in earlier experiments.
SEVERE_PROB_THRESHOLD = 0.3


# -------------------------------------------------------------------------
# Load CUSTOM CV infrastructure and feature engineering modules
# -------------------------------------------------------------------------

cv_path = (
    "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/"
    "notebooks/Cross Validator/cv.py"
)
spec = importlib.util.spec_from_file_location("cv", cv_path)
cv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cv)

FlightDelayDataLoader = cv.FlightDelayDataLoader

# Graph features module (PageRank-based features)
graph_features_path = (
    "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/notebooks/"
    "Feature Engineering/graph_features.py"
)
spec_g = importlib.util.spec_from_file_location("graph_features", graph_features_path)
graph_features = importlib.util.module_from_spec(spec_g)
spec_g.loader.exec_module(graph_features)

mlflow.autolog(disable=True)


# -------------------------------------------------------------------------
# Multi-head evaluator: metrics for NN, regressor, severe classifier
# -------------------------------------------------------------------------


class MultiHeadEvaluator:
    """
    Evaluator for multi-head models that exposes per-head metrics.

    Expected prediction columns:
        - nn_prediction_minutes : numeric NN output (10 for on-time, 75 for delay)
        - reg_prediction        : numeric regressor output (minutes)
        - severe_probability    : probability of SEVERE_DEL60 from severe head

    Labels:
        - DEP_DELAY    : continuous minutes
        - DEP_DEL15    : binary (>=15 min delay)
        - SEVERE_DEL60 : binary (>=60 min delay)

    Metrics returned:
        - nn_otpa            : OTPA accuracy (>=15)
        - reg_rmse           : RMSE from regressor head
        - reg_mae            : MAE from regressor head
        - severe_sddr        : Severe Delay Detection Rate (recall, >=60)
        - severe_sddr_prec   : Precision for severe-delay detection
    """

    def __init__(
        self,
        nn_prediction_col: str = "nn_prediction_minutes",
        reg_prediction_col: str = "reg_prediction",
        severe_prob_col: str = "severe_probability",
        numeric_label_col: str = "DEP_DELAY",
        binary_label_col: str = "DEP_DEL15",
        severe_label_col: str = "SEVERE_DEL60",
    ):
        self.nn_prediction_col = nn_prediction_col
        self.reg_prediction_col = reg_prediction_col
        self.severe_prob_col = severe_prob_col

        self.numeric_label_col = numeric_label_col
        self.binary_label_col = binary_label_col
        self.severe_label_col = severe_label_col

        # Separate evaluators for RMSE and MAE.
        self._rmse_evaluator = RegressionEvaluator(
            predictionCol=nn_prediction_col,  # overridden per call
            labelCol=numeric_label_col,
            metricName="rmse",
        )
        self._mae_evaluator = RegressionEvaluator(
            predictionCol=nn_prediction_col,  # overridden per call
            labelCol=numeric_label_col,
            metricName="mae",
        )

    def _calculate_rmse(self, predictions_df, prediction_col: str) -> float:
        clean = predictions_df.dropna(
            subset=[self.numeric_label_col, prediction_col]
        )
        self._rmse_evaluator.setPredictionCol(prediction_col)
        return float(self._rmse_evaluator.evaluate(clean)) if clean.count() > 0 else 0.0

    def _calculate_mae(self, predictions_df, prediction_col: str) -> float:
        clean = predictions_df.dropna(
            subset=[self.numeric_label_col, prediction_col]
        )
        self._mae_evaluator.setPredictionCol(prediction_col)
        return float(self._mae_evaluator.evaluate(clean)) if clean.count() > 0 else 0.0

    def _classification_metrics(
        self,
        predictions_df,
        prediction_col: str,
        threshold: float,
        label_col: str,
    ):
        """
        Generic binary classification metrics for a given prediction column
        and threshold (unweighted).
        """
        df = predictions_df.dropna(subset=[prediction_col, label_col])

        thresh_str = str(threshold).replace(".", "_")
        pred_binary_col = f"pred_binary_{prediction_col}_{thresh_str}"
        df = df.withColumn(
            pred_binary_col,
            F.when(F.col(prediction_col) >= threshold, 1).otherwise(0),
        )

        tp = df.filter((F.col(pred_binary_col) == 1) & (F.col(label_col) == 1)).count()
        fp = df.filter((F.col(pred_binary_col) == 1) & (F.col(label_col) == 0)).count()
        tn = df.filter((F.col(pred_binary_col) == 0) & (F.col(label_col) == 0)).count()
        fn = df.filter((F.col(pred_binary_col) == 0) & (F.col(label_col) == 1)).count()

        total = tp + fp + tn + fn
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        accuracy = (tp + tn) / total if total else 0.0

        return dict(
            tp=tp,
            fp=fp,
            tn=tn,
            fn=fn,
            precision=precision,
            recall=recall,
            f1=f1,
            accuracy=accuracy,
        )

    def evaluate(self, predictions_df):
        """
        Compute metrics for all three heads on a single predictions DataFrame.
        """
        metrics = {}

        # NN head (binary buckets): OTPA only
        if self.nn_prediction_col in predictions_df.columns:
            nn_otpa_stats = self._classification_metrics(
                predictions_df,
                prediction_col=self.nn_prediction_col,
                threshold=15.0,
                label_col=self.binary_label_col,
            )
            metrics.update(
                nn_otpa=nn_otpa_stats["accuracy"],
            )

        # Regressor head: RMSE + MAE
        if self.reg_prediction_col in predictions_df.columns:
            reg_rmse = self._calculate_rmse(predictions_df, self.reg_prediction_col)
            reg_mae = self._calculate_mae(predictions_df, self.reg_prediction_col)
            metrics.update(reg_rmse=reg_rmse, reg_mae=reg_mae)

        # Severe-delay classifier head: SDDR + precision (unweighted)
        if self.severe_prob_col in predictions_df.columns:
            def _extract_pos_prob(v):
                if v is None:
                    return None
                try:
                    return float(v[1])
                except Exception:
                    return None

            extract_prob_udf = F.udf(_extract_pos_prob, DoubleType())

            df_severe = predictions_df.withColumn(
                "severe_prob_scalar",
                extract_prob_udf(F.col(self.severe_prob_col)),
            )

            severe_stats = self._classification_metrics(
                df_severe,
                prediction_col="severe_prob_scalar",
                threshold=SEVERE_PROB_THRESHOLD,
                label_col=self.severe_label_col,
            )
            metrics.update(
                severe_sddr=severe_stats["recall"],
                severe_sddr_prec=severe_stats["precision"],
            )

        return metrics


# -------------------------------------------------------------------------
# Multi-head estimator (NN + severe classifier + regressor)
# -------------------------------------------------------------------------


class MultiHeadDelayEstimator:
    """
    Multi-head estimator that shares a single preprocessing pipeline and
    then trains three separate models on the same `scaled_features`:

        1) Binary on-time vs delayed NN head
            - label: delay_bucket_idx (0: <15, 1: >=15)
            - predictionCol: nn_bucket_prediction
            - numeric minutes mapping: nn_prediction_minutes

        2) Severe-delay classifier head
            - label: SEVERE_DEL60 (0/1)
            - predictionCol: severe_prediction
            - probabilityCol: severe_probability
            - no class weighting; SDDR is plain recall.

        3) Gradient-boosted tree regressor head
            - label: DEP_DELAY (continuous minutes)
            - predictionCol: reg_prediction
    """

    def __init__(self, label_col: str = "DEP_DELAY"):
        self.label_col = label_col

        # Fitted models
        self.preproc_model = None
        self.nn_model = None
        self.severe_model = None
        self.reg_model = None

        # Feature definitions (mirrors CategoricalNN_5year + lineage tweaks)
        categorical_upper = [
            "DAY_OF_WEEK",
            "OP_CARRIER",
            "ORIGIN",
            "ORIGIN_STATE_ABR",
            "DEST",
            "DEST_STATE_ABR",
            "DEP_TIME_BLK",
            "ARR_TIME_BLK",
            "DAY_OF_MONTH",
            "MONTH",
        ]
        self.categorical_features = [c.lower() for c in categorical_upper]

        numerical_upper = [
            "HOURLYPRECIPITATION",
            "HOURLYSEALEVELPRESSURE",
            "HOURLYALTIMETERSETTING",
            "HOURLYWETBULBTEMPERATURE",
            "HOURLYSTATIONPRESSURE",
            "HOURLYWINDDIRECTION",
            "HOURLYRELATIVEHUMIDITY",
            "HOURLYWINDSPEED",
            "HOURLYDEWPOINTTEMPERATURE",
            "HOURLYDRYBULBTEMPERATURE",
            "HOURLYVISIBILITY",
            "CRS_ELAPSED_TIME",
            "DISTANCE",
            "ELEVATION",
            "ORIGIN_PAGERANK_WEIGHTED",
            "ORIGIN_PAGERANK_UNWEIGHTED",
            "DEST_PAGERANK_WEIGHTED",
            "DEST_PAGERANK_UNWEIGHTED",
        ]
        self.numerical_features = [c.lower() for c in numerical_upper]

        # Visibility-derived flags
        self.numerical_features.extend(
            ["visibility_missing_flag", "very_bad_vis_flag", "bad_vis_flag"]
        )

        # Precipitation-derived flags
        self.numerical_features.extend(
            ["precip_missing_flag", "any_precip_flag", "heavy_precip_flag"]
        )

        # Wind-derived flags
        self.numerical_features.extend(
            ["wind_missing_flag", "high_wind_flag", "very_high_wind_flag"]
        )

    # -------------------------- Internal helpers --------------------------

    def _prepare(self, df):
        """
        Prepare DataFrame for modeling (shared for all heads).
        """
        # 1. Cast label to double and filter invalid rows
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # 1a. Ensure binary labels used by downstream evaluators are present.
        if "DEP_DEL15" not in df.columns:
            df = df.withColumn(
                "DEP_DEL15", when(col(self.label_col) >= 15, 1).otherwise(0)
            )
        if "SEVERE_DEL60" not in df.columns:
            df = df.withColumn(
                "SEVERE_DEL60", when(col(self.label_col) >= 60, 1).otherwise(0)
            )

        # 2. Clean numerical features (regex-strip non-numeric, cast to double)
        for f in self.numerical_features:
            if f in df.columns:
                df = df.withColumn(
                    f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", "")
                )
                df = df.withColumn(
                    f, when(length(trim(col(f))) == 0, None).otherwise(col(f))
                )
                df = df.withColumn(f, col(f).cast(DoubleType()))

        # 2a. Visibility flags
        if "hourlyvisibility" in df.columns:
            df = df.withColumn(
                "visibility_missing_flag",
                when(col("hourlyvisibility").isNull(), 1.0).otherwise(0.0),
            )
            df = df.withColumn(
                "very_bad_vis_flag",
                when(col("hourlyvisibility").isNull(), None)
                .otherwise(when(col("hourlyvisibility") < 2, 1.0).otherwise(0.0)),
            )
            df = df.withColumn(
                "bad_vis_flag",
                when(col("hourlyvisibility").isNull(), None)
                .otherwise(when(col("hourlyvisibility") < 4, 1.0).otherwise(0.0)),
            )

        # 2b. Precipitation flags
        if "hourlyprecipitation" in df.columns:
            df = df.withColumn(
                "precip_missing_flag",
                when(col("hourlyprecipitation").isNull(), 1.0).otherwise(0.0),
            )
            df = df.withColumn(
                "any_precip_flag",
                when(col("hourlyprecipitation").isNull(), None)
                .otherwise(
                    when(col("hourlyprecipitation") > 0.01, 1.0).otherwise(0.0)
                ),
            )
            df = df.withColumn(
                "heavy_precip_flag",
                when(col("hourlyprecipitation").isNull(), None)
                .otherwise(
                    when(col("hourlyprecipitation") >= 0.1, 1.0).otherwise(0.0)
                ),
            )

        # 2c. Wind flags
        if "hourlywindspeed" in df.columns:
            df = df.withColumn(
                "wind_missing_flag",
                when(col("hourlywindspeed").isNull(), 1.0).otherwise(0.0),
            )
            df = df.withColumn(
                "high_wind_flag",
                when(col("hourlywindspeed").isNull(), None)
                .otherwise(when(col("hourlywindspeed") > 20, 1.0).otherwise(0.0)),
            )
            df = df.withColumn(
                "very_high_wind_flag",
                when(col("hourlywindspeed").isNull(), None)
                .otherwise(when(col("hourlywindspeed") > 25, 1.0).otherwise(0.0)),
            )

        # 2d. Cleaned categorical columns for StringIndexer
        for f in self.categorical_features:
            if f in df.columns:
                clean_col = f"{f}_clean"
                if f in ["day_of_week", "month", "dep_time_blk", "arr_time_blk", "day_of_month"]:
                    df = df.withColumn(
                        clean_col,
                        when(col(f).isNull(), "UNKNOWN").otherwise(
                            col(f).cast(StringType())
                        ),
                    )
                else:
                    df = df.withColumn(
                        clean_col,
                        when(col(f).isNull(), "UNKNOWN").otherwise(col(f)),
                    )

        # 3. Derive binary delay bucket label: on-time (<15) vs delayed (>=15)
        delay = col(self.label_col)
        df = df.withColumn("delay_bucket_idx", when(delay < 15, 0).otherwise(1))

        return df

    def _build_preprocessing_pipeline(self, df):
        """
        Shared preprocessing pipeline:
            - GraphFeaturesEstimator
            - Median imputation for numeric features
            - StringIndexer for categoricals
            - VectorAssembler + StandardScaler -> scaled_features
        """
        stages = []

        # 0. Graph features
        graph_estimator = graph_features.GraphFeaturesEstimator(
            origin_col="origin",
            dest_col="dest",
            reset_probability=0.15,
            max_iter=10,
        )
        stages.append(graph_estimator)

        # 1. Median imputation for numeric features
        imputers = [
            Imputer(inputCols=[f], outputCols=[f"{f}_imputed"], strategy="median")
            for f in self.numerical_features
            if f in df.columns
        ]
        stages.extend(imputers)

        # 2. Categorical encoding via StringIndexer
        indexed_cat_features = []
        for f in self.categorical_features:
            if f in df.columns:
                idx_col = f"{f}_idx"
                stages.append(
                    StringIndexer(
                        inputCol=f"{f}_clean", outputCol=idx_col, handleInvalid="keep"
                    )
                )
                indexed_cat_features.append(idx_col)

        # 3. Assemble features
        numeric_inputs = [
            f"{f}_imputed" for f in self.numerical_features if f in df.columns
        ]
        feature_inputs = numeric_inputs + indexed_cat_features

        assembler = VectorAssembler(
            inputCols=feature_inputs, outputCol="features", handleInvalid="skip"
        )

        # 4. Standardize features
        scaler = StandardScaler(
            inputCol="features", outputCol="scaled_features", withStd=True, withMean=True
        )

        stages.extend([assembler, scaler])
        pipeline = Pipeline(stages=stages)
        return pipeline, feature_inputs

    # ---------------------------- Public API ------------------------------

    def fit(self, df):
        """
        Fit all three heads (NN, severe classifier, regressor) on the same
        preprocessed features.
        """
        df_prep = self._prepare(df)
        preproc_pipeline, feature_inputs = self._build_preprocessing_pipeline(df_prep)

        num_input = len(feature_inputs)
        if num_input == 0:
            raise ValueError(
                "No input features found for MultiHeadDelayEstimator. "
                f"Checked: {feature_inputs}"
            )

        self.preproc_model = preproc_pipeline.fit(df_prep)
        df_feats = self.preproc_model.transform(df_prep)

        # 1. Binary NN head
        nn_layers = [num_input, 32, 16, 2]
        nn_clf = MultilayerPerceptronClassifier(
            featuresCol="scaled_features",
            labelCol="delay_bucket_idx",
            predictionCol="nn_bucket_prediction",
            probabilityCol="nn_bucket_probability",
            layers=nn_layers,
            maxIter=NN_MAX_ITER,
            seed=42,
        )
        self.nn_model = nn_clf.fit(df_feats)

        # 2. Severe-delay classifier head (SEVERE_DEL60), unweighted
        severe_clf = LogisticRegression(
            featuresCol="scaled_features",
            labelCol="SEVERE_DEL60",
            predictionCol="severe_prediction",
            probabilityCol="severe_probability",
            rawPredictionCol="severe_raw_prediction",
            maxIter=50,
            regParam=0.01,
            elasticNetParam=0.0,
        )
        self.severe_model = severe_clf.fit(df_feats)

        # 3. Gradient-boosted tree regressor head (same best config as earlier)
        reg = GBTRegressor(
            featuresCol="scaled_features",
            labelCol=self.label_col,
            predictionCol="reg_prediction",
            maxDepth=5,
            maxIter=50,
            stepSize=0.1,
            seed=42,
        )
        self.reg_model = reg.fit(df_feats)

        return self

    def transform(self, df):
        """
        Transform a DataFrame using all fitted heads.
        """
        if (
            self.preproc_model is None
            or self.nn_model is None
            or self.severe_model is None
            or self.reg_model is None
        ):
            raise RuntimeError(
                "MultiHeadDelayEstimator must be fit() before transform()."
            )

        df_prep = self._prepare(df)
        df_feats = self.preproc_model.transform(df_prep)

        # Apply heads sequentially so all predictions end up in one DataFrame
        preds = self.nn_model.transform(df_feats)
        preds = self.severe_model.transform(preds)
        preds = self.reg_model.transform(preds)

        # Map NN bucket prediction (0/1) to a pseudo-minute value
        bucket = col("nn_bucket_prediction")
        preds = preds.withColumn(
            "nn_prediction_minutes",
            when(bucket == 0, 10.0).otherwise(75.0),
        )

        # Primary numeric prediction for legacy-style metrics
        preds = preds.withColumn("prediction", col("nn_prediction_minutes"))
        return preds


# -------------------------------------------------------------------------
# Multi-head CV wrapper reporting OTPA, RMSE, MAE, SDDR, precision
# -------------------------------------------------------------------------


class MultiHeadCVBonus:
    """
    Cross-validator for multi-head models using CUSTOM cv.py folds.

    - Uses FlightDelayDataLoader (3M / 12M / 60M) to load pre-built folds.
    - Evaluates:
          * nn_otpa
          * reg_rmse
          * reg_mae
          * severe_sddr
          * severe_sddr_prec
      on TRAIN and VAL for each CV fold.
    - Returns a pandas DataFrame with per-fold rows plus mean/std rows
      for Train and Val separately.
    - `evaluate()` trains on the last fold's train set and evaluates TRAIN / TEST,
      returning another DataFrame.
    """

    def __init__(self, estimator, version, dataloader=None):
        self.estimator = estimator
        self.version = version

        if dataloader is not None:
            self.data_loader = dataloader
        else:
            self.data_loader = FlightDelayDataLoader()
            self.data_loader.load()

        self.folds = self.data_loader.get_version(version)
        self.evaluator = MultiHeadEvaluator()

        self.models = []
        self.metrics = []
        self.test_model = None
        self.test_metric = None

    def fit(self):
        """
        Run CV on all folds except the last (held-out test).

        Returns:
            pandas.DataFrame with rows:
                Fold 1 Train, Fold 1 Val, ..., Mean Train, Std Train,
                Mean Val, Std Val
        """
        results = []

        for i, (train_df, val_df) in enumerate(self.folds[:-1]):
            fold_name = f"Fold {i + 1}"

            model = self.estimator.fit(train_df)

            # Evaluate on training set
            train_preds = model.transform(train_df)
            train_metric = self.evaluator.evaluate(train_preds)

            # Evaluate on validation set
            val_preds = model.transform(val_df)
            val_metric = self.evaluator.evaluate(val_preds)

            self.metrics.append(val_metric)
            self.models.append(model)

            results.append(
                {
                    "fold": fold_name,
                    **{f"{k}_train": v for k, v in train_metric.items()},
                    **{f"{k}_val": v for k, v in val_metric.items()},
                }
            )

        # Build alternating Train/Val rows
        rows = []
        metric_keys = ["nn_otpa", "reg_rmse", "reg_mae", "severe_sddr", "severe_sddr_prec"]
        for res in results:
            fold = res["fold"]
            # Train row
            rows.append(
                {
                    "Fold": f"{fold} Train",
                    **{k: res[f"{k}_train"] for k in metric_keys},
                }
            )
            # Val row
            rows.append(
                {
                    "Fold": f"{fold} Val",
                    **{k: res[f"{k}_val"] for k in metric_keys},
                }
            )

        m = pd.DataFrame(rows)

        # Mean / std per split
        train_rows = m[m["Fold"].str.contains("Train")]
        val_rows = m[m["Fold"].str.contains("Val")]

        mean_train = {
            "Fold": "Mean Train",
            **{k: train_rows[k].mean() for k in metric_keys},
        }
        std_train = {
            "Fold": "Std Train",
            **{k: train_rows[k].std() for k in metric_keys},
        }
        mean_val = {
            "Fold": "Mean Val",
            **{k: val_rows[k].mean() for k in metric_keys},
        }
        std_val = {
            "Fold": "Std Val",
            **{k: val_rows[k].std() for k in metric_keys},
        }

        summary = pd.DataFrame([mean_train, std_train, mean_val, std_val])
        m = pd.concat([m, summary], ignore_index=True)
        return m

    def evaluate(self):
        """
        Train on the last fold's train set and evaluate on TRAIN and TEST.

        Returns:
            pandas.DataFrame with 2 rows: Train, Test.
        """
        train_df, test_df = self.folds[-1]
        self.test_model = self.estimator.fit(train_df)

        # Evaluate on training set
        train_preds = self.test_model.transform(train_df)
        train_metric = self.evaluator.evaluate(train_preds)

        # Evaluate on test set
        test_preds = self.test_model.transform(test_df)
        test_metric = self.evaluator.evaluate(test_preds)

        self.test_metric = test_metric

        rows = [
            {"Split": "Train", **train_metric},
            {"Split": "Test", **test_metric},
        ]
        return pd.DataFrame(rows)

In [0]:
if __name__ == "__main__":
    loader = FlightDelayDataLoader()
    loader.load()

    est = MultiHeadDelayEstimator(label_col="DEP_DELAY")

    cv_set = MultiHeadCVBonus(
        estimator=est,
        version="60M",  # or "3M"/"12M" for smaller experiments
        dataloader=loader,
    )

    cv_metrics = cv_set.fit()
    print("Cross-validation metrics (Train / Val per fold + mean/std):")
    print(cv_metrics)

    test_metrics = cv_set.evaluate()
    print("Held-out TRAIN / TEST metrics:")
    print(test_metrics)

In [0]:
    sc = spark.sparkContext  # uses the existing Spark session in Databricks

    # Get number of executors (exclude driver node)
    num_executors = len(sc._jsc.sc().statusTracker().getExecutorInfos()) - 1

    # Get executor configuration from Spark conf
    executor_memory = sc.getConf().get("spark.executor.memory", "Unknown")
    executor_cores = sc.getConf().get("spark.executor.cores", "Unknown")
    actual_cores = (
        sc.defaultParallelism / num_executors if num_executors > 0 else "Unknown"
    )

    print(f"Number of Executors: {num_executors}")
    print(f"Executor Memory: {executor_memory}")
    print(f"Executor Cores: {executor_cores}")
    print(
        f"Estimated Cores per Executor: "
        f"{int(actual_cores) if isinstance(actual_cores, float) else actual_cores}"
    )
    print(f"Total Parallelism: {sc.defaultParallelism}")
    print("=" * 80)

## Updated the SDDR model to output weighted recall, which will hopefully help identify severe delays correctly

In [0]:
"""
Bonus: Multi-Head delay modeling configuration.

This file defines a multi-head model using:
    - The best-performing NN and GBT settings from prior iterations
      (strong OTPA and RMSE).
    - A "balanced" severe-delay head that is *trained* with a simple
      positive class weight but *evaluated* with plain (unweighted)
      SDDR and precision at a moderate threshold.

Heads:
    1) Binary NN classifier for OTPA (DEP_DELAY < 15 vs >= 15)
    2) GBTRegressor for continuous RMSE (DEP_DELAY)
    3) LogisticRegression for severe delays (SEVERE_DEL60) with
       class-weighted training and unweighted recall/precision,
       threshold ~= 0.3

The cross-validator reports, per fold:
    - nn_otpa
    - reg_rmse
    - reg_mae
    - severe_sddr
    - severe_sddr_prec
"""

import importlib.util

from pyspark.sql import functions as F
from pyspark.sql.functions import col, isnan, regexp_replace, when, length, trim
from pyspark.sql.types import DoubleType, StringType

from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier, LogisticRegression
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import pandas as pd


# -------------------------------------------------------------------------
# Config: head behavior
# -------------------------------------------------------------------------

# NN OTPA head: keep architecture [input, 32, 16, 2] with 50 iterations,
# which performed best in prior 5-year runs.
NN_MAX_ITER = 50

# Severe-delay head: probability threshold for classifying a flight as
# "severe delay" in the evaluator. Using a moderate value here (~0.3)
# gave a more balanced SDDR in earlier experiments.
SEVERE_PROB_THRESHOLD = 0.3


# -------------------------------------------------------------------------
# Load CUSTOM CV infrastructure and feature engineering modules
# -------------------------------------------------------------------------

cv_path = (
    "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/"
    "notebooks/Cross Validator/cv.py"
)
spec = importlib.util.spec_from_file_location("cv", cv_path)
cv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cv)

FlightDelayDataLoader = cv.FlightDelayDataLoader

# Graph features module (PageRank-based features)
graph_features_path = (
    "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/notebooks/"
    "Feature Engineering/graph_features.py"
)
spec_g = importlib.util.spec_from_file_location("graph_features", graph_features_path)
graph_features = importlib.util.module_from_spec(spec_g)
spec_g.loader.exec_module(graph_features)

mlflow.autolog(disable=True)


# -------------------------------------------------------------------------
# Multi-head evaluator: metrics for NN, regressor, severe classifier
# -------------------------------------------------------------------------


class MultiHeadEvaluator:
    """
    Evaluator for multi-head models that exposes per-head metrics.

    Expected prediction columns:
        - nn_prediction_minutes : numeric NN output (10 for on-time, 75 for delay)
        - reg_prediction        : numeric regressor output (minutes)
        - severe_probability    : probability of SEVERE_DEL60 from severe head

    Labels:
        - DEP_DELAY    : continuous minutes
        - DEP_DEL15    : binary (>=15 min delay)
        - SEVERE_DEL60 : binary (>=60 min delay)

    Metrics returned:
        - nn_otpa            : OTPA accuracy (>=15)
        - reg_rmse           : RMSE from regressor head
        - reg_mae            : MAE from regressor head
        - severe_sddr        : Severe Delay Detection Rate (recall, >=60)
        - severe_sddr_prec   : Precision for severe-delay detection
    """

    def __init__(
        self,
        nn_prediction_col: str = "nn_prediction_minutes",
        reg_prediction_col: str = "reg_prediction",
        severe_prob_col: str = "severe_probability",
        numeric_label_col: str = "DEP_DELAY",
        binary_label_col: str = "DEP_DEL15",
        severe_label_col: str = "SEVERE_DEL60",
    ):
        self.nn_prediction_col = nn_prediction_col
        self.reg_prediction_col = reg_prediction_col
        self.severe_prob_col = severe_prob_col

        self.numeric_label_col = numeric_label_col
        self.binary_label_col = binary_label_col
        self.severe_label_col = severe_label_col

        # Separate evaluators for RMSE and MAE.
        self._rmse_evaluator = RegressionEvaluator(
            predictionCol=nn_prediction_col,  # overridden per call
            labelCol=numeric_label_col,
            metricName="rmse",
        )
        self._mae_evaluator = RegressionEvaluator(
            predictionCol=nn_prediction_col,  # overridden per call
            labelCol=numeric_label_col,
            metricName="mae",
        )

    def _calculate_rmse(self, predictions_df, prediction_col: str) -> float:
        clean = predictions_df.dropna(
            subset=[self.numeric_label_col, prediction_col]
        )
        self._rmse_evaluator.setPredictionCol(prediction_col)
        return float(self._rmse_evaluator.evaluate(clean)) if clean.count() > 0 else 0.0

    def _calculate_mae(self, predictions_df, prediction_col: str) -> float:
        clean = predictions_df.dropna(
            subset=[self.numeric_label_col, prediction_col]
        )
        self._mae_evaluator.setPredictionCol(prediction_col)
        return float(self._mae_evaluator.evaluate(clean)) if clean.count() > 0 else 0.0

    def _classification_metrics(
        self,
        predictions_df,
        prediction_col: str,
        threshold: float,
        label_col: str,
    ):
        """
        Generic binary classification metrics for a given prediction column
        and threshold (unweighted).
        """
        df = predictions_df.dropna(subset=[prediction_col, label_col])

        thresh_str = str(threshold).replace(".", "_")
        pred_binary_col = f"pred_binary_{prediction_col}_{thresh_str}"
        df = df.withColumn(
            pred_binary_col,
            F.when(F.col(prediction_col) >= threshold, 1).otherwise(0),
        )

        tp = df.filter((F.col(pred_binary_col) == 1) & (F.col(label_col) == 1)).count()
        fp = df.filter((F.col(pred_binary_col) == 1) & (F.col(label_col) == 0)).count()
        tn = df.filter((F.col(pred_binary_col) == 0) & (F.col(label_col) == 0)).count()
        fn = df.filter((F.col(pred_binary_col) == 0) & (F.col(label_col) == 1)).count()

        total = tp + fp + tn + fn
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        accuracy = (tp + tn) / total if total else 0.0

        return dict(
            tp=tp,
            fp=fp,
            tn=tn,
            fn=fn,
            precision=precision,
            recall=recall,
            f1=f1,
            accuracy=accuracy,
        )

    def evaluate(self, predictions_df):
        """
        Compute metrics for all three heads on a single predictions DataFrame.
        """
        metrics = {}

        # NN head (binary buckets): OTPA only
        if self.nn_prediction_col in predictions_df.columns:
            nn_otpa_stats = self._classification_metrics(
                predictions_df,
                prediction_col=self.nn_prediction_col,
                threshold=15.0,
                label_col=self.binary_label_col,
            )
            metrics.update(
                nn_otpa=nn_otpa_stats["accuracy"],
            )

        # Regressor head: RMSE + MAE
        if self.reg_prediction_col in predictions_df.columns:
            reg_rmse = self._calculate_rmse(predictions_df, self.reg_prediction_col)
            reg_mae = self._calculate_mae(predictions_df, self.reg_prediction_col)
            metrics.update(reg_rmse=reg_rmse, reg_mae=reg_mae)

        # Severe-delay classifier head: SDDR + precision (unweighted
        # metrics, even though the LR is trained with class weights).
        if self.severe_prob_col in predictions_df.columns:
            def _extract_pos_prob(v):
                if v is None:
                    return None
                try:
                    return float(v[1])
                except Exception:
                    return None

            extract_prob_udf = F.udf(_extract_pos_prob, DoubleType())

            df_severe = predictions_df.withColumn(
                "severe_prob_scalar",
                extract_prob_udf(F.col(self.severe_prob_col)),
            )

            severe_stats = self._classification_metrics(
                df_severe,
                prediction_col="severe_prob_scalar",
                threshold=SEVERE_PROB_THRESHOLD,
                label_col=self.severe_label_col,
            )
            metrics.update(
                severe_sddr=severe_stats["recall"],
                severe_sddr_prec=severe_stats["precision"],
            )

        return metrics


# -------------------------------------------------------------------------
# Multi-head estimator (NN + severe classifier + regressor)
# -------------------------------------------------------------------------


class MultiHeadDelayEstimator:
    """
    Multi-head estimator that shares a single preprocessing pipeline and
    then trains three separate models on the same `scaled_features`:

        1) Binary on-time vs delayed NN head
            - label: delay_bucket_idx (0: <15, 1: >=15)
            - predictionCol: nn_bucket_prediction
            - numeric minutes mapping: nn_prediction_minutes

        2) Severe-delay classifier head
            - label: SEVERE_DEL60 (0/1)
            - predictionCol: severe_prediction
            - probabilityCol: severe_probability
            - no class weighting; SDDR is plain recall.

        3) Gradient-boosted tree regressor head
            - label: DEP_DELAY (continuous minutes)
            - predictionCol: reg_prediction
    """

    def __init__(self, label_col: str = "DEP_DELAY"):
        self.label_col = label_col

        # Fitted models
        self.preproc_model = None
        self.nn_model = None
        self.severe_model = None
        self.reg_model = None

        # Feature definitions (mirrors CategoricalNN_5year + lineage tweaks)
        categorical_upper = [
            "DAY_OF_WEEK",
            "OP_CARRIER",
            "ORIGIN",
            "ORIGIN_STATE_ABR",
            "DEST",
            "DEST_STATE_ABR",
            "DEP_TIME_BLK",
            "ARR_TIME_BLK",
            "DAY_OF_MONTH",
            "MONTH",
        ]
        self.categorical_features = [c.lower() for c in categorical_upper]

        numerical_upper = [
            "HOURLYPRECIPITATION",
            "HOURLYSEALEVELPRESSURE",
            "HOURLYALTIMETERSETTING",
            "HOURLYWETBULBTEMPERATURE",
            "HOURLYSTATIONPRESSURE",
            "HOURLYWINDDIRECTION",
            "HOURLYRELATIVEHUMIDITY",
            "HOURLYWINDSPEED",
            "HOURLYDEWPOINTTEMPERATURE",
            "HOURLYDRYBULBTEMPERATURE",
            "HOURLYVISIBILITY",
            "CRS_ELAPSED_TIME",
            "DISTANCE",
            "ELEVATION",
            "ORIGIN_PAGERANK_WEIGHTED",
            "ORIGIN_PAGERANK_UNWEIGHTED",
            "DEST_PAGERANK_WEIGHTED",
            "DEST_PAGERANK_UNWEIGHTED",
        ]
        self.numerical_features = [c.lower() for c in numerical_upper]

        # Visibility-derived flags
        self.numerical_features.extend(
            ["visibility_missing_flag", "very_bad_vis_flag", "bad_vis_flag"]
        )

        # Precipitation-derived flags
        self.numerical_features.extend(
            ["precip_missing_flag", "any_precip_flag", "heavy_precip_flag"]
        )

        # Wind-derived flags
        self.numerical_features.extend(
            ["wind_missing_flag", "high_wind_flag", "very_high_wind_flag"]
        )

    # -------------------------- Internal helpers --------------------------

    def _prepare(self, df):
        """
        Prepare DataFrame for modeling (shared for all heads).
        """
        # 1. Cast label to double and filter invalid rows
        df = df.withColumn(self.label_col, col(self.label_col).cast(DoubleType()))
        df = df.filter(~(col(self.label_col).isNull() | isnan(col(self.label_col))))

        # 1a. Ensure binary labels used by downstream evaluators are present.
        if "DEP_DEL15" not in df.columns:
            df = df.withColumn(
                "DEP_DEL15", when(col(self.label_col) >= 15, 1).otherwise(0)
            )
        if "SEVERE_DEL60" not in df.columns:
            df = df.withColumn(
                "SEVERE_DEL60", when(col(self.label_col) >= 60, 1).otherwise(0)
            )

        # 2. Clean numerical features (regex-strip non-numeric, cast to double)
        for f in self.numerical_features:
            if f in df.columns:
                df = df.withColumn(
                    f, regexp_replace(col(f).cast(StringType()), r"[^0-9+\-\.]", "")
                )
                df = df.withColumn(
                    f, when(length(trim(col(f))) == 0, None).otherwise(col(f))
                )
                df = df.withColumn(f, col(f).cast(DoubleType()))

        # 2a. Visibility flags
        if "hourlyvisibility" in df.columns:
            df = df.withColumn(
                "visibility_missing_flag",
                when(col("hourlyvisibility").isNull(), 1.0).otherwise(0.0),
            )
            df = df.withColumn(
                "very_bad_vis_flag",
                when(col("hourlyvisibility").isNull(), None)
                .otherwise(when(col("hourlyvisibility") < 2, 1.0).otherwise(0.0)),
            )
            df = df.withColumn(
                "bad_vis_flag",
                when(col("hourlyvisibility").isNull(), None)
                .otherwise(when(col("hourlyvisibility") < 4, 1.0).otherwise(0.0)),
            )

        # 2b. Precipitation flags
        if "hourlyprecipitation" in df.columns:
            df = df.withColumn(
                "precip_missing_flag",
                when(col("hourlyprecipitation").isNull(), 1.0).otherwise(0.0),
            )
            df = df.withColumn(
                "any_precip_flag",
                when(col("hourlyprecipitation").isNull(), None)
                .otherwise(
                    when(col("hourlyprecipitation") > 0.01, 1.0).otherwise(0.0)
                ),
            )
            df = df.withColumn(
                "heavy_precip_flag",
                when(col("hourlyprecipitation").isNull(), None)
                .otherwise(
                    when(col("hourlyprecipitation") >= 0.1, 1.0).otherwise(0.0)
                ),
            )

        # 2c. Wind flags
        if "hourlywindspeed" in df.columns:
            df = df.withColumn(
                "wind_missing_flag",
                when(col("hourlywindspeed").isNull(), 1.0).otherwise(0.0),
            )
            df = df.withColumn(
                "high_wind_flag",
                when(col("hourlywindspeed").isNull(), None)
                .otherwise(when(col("hourlywindspeed") > 20, 1.0).otherwise(0.0)),
            )
            df = df.withColumn(
                "very_high_wind_flag",
                when(col("hourlywindspeed").isNull(), None)
                .otherwise(when(col("hourlywindspeed") > 25, 1.0).otherwise(0.0)),
            )

        # 2d. Cleaned categorical columns for StringIndexer
        for f in self.categorical_features:
            if f in df.columns:
                clean_col = f"{f}_clean"
                if f in ["day_of_week", "month", "dep_time_blk", "arr_time_blk", "day_of_month"]:
                    df = df.withColumn(
                        clean_col,
                        when(col(f).isNull(), "UNKNOWN").otherwise(
                            col(f).cast(StringType())
                        ),
                    )
                else:
                    df = df.withColumn(
                        clean_col,
                        when(col(f).isNull(), "UNKNOWN").otherwise(col(f)),
                    )

        # 3. Derive binary delay bucket label: on-time (<15) vs delayed (>=15)
        delay = col(self.label_col)
        df = df.withColumn("delay_bucket_idx", when(delay < 15, 0).otherwise(1))

        return df

    def _build_preprocessing_pipeline(self, df):
        """
        Shared preprocessing pipeline:
            - GraphFeaturesEstimator
            - Median imputation for numeric features
            - StringIndexer for categoricals
            - VectorAssembler + StandardScaler -> scaled_features
        """
        stages = []

        # 0. Graph features
        graph_estimator = graph_features.GraphFeaturesEstimator(
            origin_col="origin",
            dest_col="dest",
            reset_probability=0.15,
            max_iter=10,
        )
        stages.append(graph_estimator)

        # 1. Median imputation for numeric features
        imputers = [
            Imputer(inputCols=[f], outputCols=[f"{f}_imputed"], strategy="median")
            for f in self.numerical_features
            if f in df.columns
        ]
        stages.extend(imputers)

        # 2. Categorical encoding via StringIndexer
        indexed_cat_features = []
        for f in self.categorical_features:
            if f in df.columns:
                idx_col = f"{f}_idx"
                stages.append(
                    StringIndexer(
                        inputCol=f"{f}_clean", outputCol=idx_col, handleInvalid="keep"
                    )
                )
                indexed_cat_features.append(idx_col)

        # 3. Assemble features
        numeric_inputs = [
            f"{f}_imputed" for f in self.numerical_features if f in df.columns
        ]
        feature_inputs = numeric_inputs + indexed_cat_features

        assembler = VectorAssembler(
            inputCols=feature_inputs, outputCol="features", handleInvalid="skip"
        )

        # 4. Standardize features
        scaler = StandardScaler(
            inputCol="features", outputCol="scaled_features", withStd=True, withMean=True
        )

        stages.extend([assembler, scaler])
        pipeline = Pipeline(stages=stages)
        return pipeline, feature_inputs

    # ---------------------------- Public API ------------------------------

    def fit(self, df):
        """
        Fit all three heads (NN, severe classifier, regressor) on the same
        preprocessed features.
        """
        df_prep = self._prepare(df)
        preproc_pipeline, feature_inputs = self._build_preprocessing_pipeline(df_prep)

        num_input = len(feature_inputs)
        if num_input == 0:
            raise ValueError(
                "No input features found for MultiHeadDelayEstimator. "
                f"Checked: {feature_inputs}"
            )

        self.preproc_model = preproc_pipeline.fit(df_prep)
        df_feats = self.preproc_model.transform(df_prep)

        # 1. Binary NN head
        nn_layers = [num_input, 32, 16, 2]
        nn_clf = MultilayerPerceptronClassifier(
            featuresCol="scaled_features",
            labelCol="delay_bucket_idx",
            predictionCol="nn_bucket_prediction",
            probabilityCol="nn_bucket_probability",
            layers=nn_layers,
            maxIter=NN_MAX_ITER,
            seed=42,
        )
        self.nn_model = nn_clf.fit(df_feats)

        # 2. Severe-delay classifier head (SEVERE_DEL60), with simple
        # class weighting to approximate the earlier "balanced" setup.
        class_counts = df_feats.groupBy("SEVERE_DEL60").count().collect()
        counts = {row["SEVERE_DEL60"]: row["count"] for row in class_counts}
        severe_count = counts.get(1, 0)
        nonsevere_count = counts.get(0, 0)

        pos_weight = (
            float(nonsevere_count) / severe_count if severe_count > 0 else 1.0
        )

        df_feats_severe = df_feats.withColumn(
            "severe_weight",
            when(col("SEVERE_DEL60") == 1, pos_weight).otherwise(1.0),
        )

        severe_clf = LogisticRegression(
            featuresCol="scaled_features",
            labelCol="SEVERE_DEL60",
            weightCol="severe_weight",
            predictionCol="severe_prediction",
            probabilityCol="severe_probability",
            rawPredictionCol="severe_raw_prediction",
            maxIter=50,
            regParam=0.01,
            elasticNetParam=0.0,
        )
        self.severe_model = severe_clf.fit(df_feats_severe)

        # 3. Gradient-boosted tree regressor head (same best config as earlier)
        reg = GBTRegressor(
            featuresCol="scaled_features",
            labelCol=self.label_col,
            predictionCol="reg_prediction",
            maxDepth=5,
            maxIter=50,
            stepSize=0.1,
            seed=42,
        )
        self.reg_model = reg.fit(df_feats)

        return self

    def transform(self, df):
        """
        Transform a DataFrame using all fitted heads.
        """
        if (
            self.preproc_model is None
            or self.nn_model is None
            or self.severe_model is None
            or self.reg_model is None
        ):
            raise RuntimeError(
                "MultiHeadDelayEstimator must be fit() before transform()."
            )

        df_prep = self._prepare(df)
        df_feats = self.preproc_model.transform(df_prep)

        # Apply heads sequentially so all predictions end up in one DataFrame
        preds = self.nn_model.transform(df_feats)
        preds = self.severe_model.transform(preds)
        preds = self.reg_model.transform(preds)

        # Map NN bucket prediction (0/1) to a pseudo-minute value
        bucket = col("nn_bucket_prediction")
        preds = preds.withColumn(
            "nn_prediction_minutes",
            when(bucket == 0, 10.0).otherwise(75.0),
        )

        # Primary numeric prediction for legacy-style metrics
        preds = preds.withColumn("prediction", col("nn_prediction_minutes"))
        return preds


# -------------------------------------------------------------------------
# Multi-head CV wrapper reporting OTPA, RMSE, MAE, SDDR, precision
# -------------------------------------------------------------------------


class MultiHeadCVBonus:
    """
    Cross-validator for multi-head models using CUSTOM cv.py folds.

    - Uses FlightDelayDataLoader (3M / 12M / 60M) to load pre-built folds.
    - Evaluates:
          * nn_otpa
          * reg_rmse
          * reg_mae
          * severe_sddr
          * severe_sddr_prec
      on TRAIN and VAL for each CV fold.
    - Returns a pandas DataFrame with per-fold rows plus mean/std rows
      for Train and Val separately.
    - `evaluate()` trains on the last fold's train set and evaluates TRAIN / TEST,
      returning another DataFrame.
    """

    def __init__(self, estimator, version, dataloader=None):
        self.estimator = estimator
        self.version = version

        if dataloader is not None:
            self.data_loader = dataloader
        else:
            self.data_loader = FlightDelayDataLoader()
            self.data_loader.load()

        self.folds = self.data_loader.get_version(version)
        self.evaluator = MultiHeadEvaluator()

        self.models = []
        self.metrics = []
        self.test_model = None
        self.test_metric = None

    def fit(self):
        """
        Run CV on all folds except the last (held-out test).

        Returns:
            pandas.DataFrame with rows:
                Fold 1 Train, Fold 1 Val, ..., Mean Train, Std Train,
                Mean Val, Std Val
        """
        results = []

        for i, (train_df, val_df) in enumerate(self.folds[:-1]):
            fold_name = f"Fold {i + 1}"

            model = self.estimator.fit(train_df)

            # Evaluate on training set
            train_preds = model.transform(train_df)
            train_metric = self.evaluator.evaluate(train_preds)

            # Evaluate on validation set
            val_preds = model.transform(val_df)
            val_metric = self.evaluator.evaluate(val_preds)

            self.metrics.append(val_metric)
            self.models.append(model)

            results.append(
                {
                    "fold": fold_name,
                    **{f"{k}_train": v for k, v in train_metric.items()},
                    **{f"{k}_val": v for k, v in val_metric.items()},
                }
            )

        # Build alternating Train/Val rows
        rows = []
        metric_keys = ["nn_otpa", "reg_rmse", "reg_mae", "severe_sddr", "severe_sddr_prec"]
        for res in results:
            fold = res["fold"]
            # Train row
            rows.append(
                {
                    "Fold": f"{fold} Train",
                    **{k: res[f"{k}_train"] for k in metric_keys},
                }
            )
            # Val row
            rows.append(
                {
                    "Fold": f"{fold} Val",
                    **{k: res[f"{k}_val"] for k in metric_keys},
                }
            )

        m = pd.DataFrame(rows)

        # Mean / std per split
        train_rows = m[m["Fold"].str.contains("Train")]
        val_rows = m[m["Fold"].str.contains("Val")]

        mean_train = {
            "Fold": "Mean Train",
            **{k: train_rows[k].mean() for k in metric_keys},
        }
        std_train = {
            "Fold": "Std Train",
            **{k: train_rows[k].std() for k in metric_keys},
        }
        mean_val = {
            "Fold": "Mean Val",
            **{k: val_rows[k].mean() for k in metric_keys},
        }
        std_val = {
            "Fold": "Std Val",
            **{k: val_rows[k].std() for k in metric_keys},
        }

        summary = pd.DataFrame([mean_train, std_train, mean_val, std_val])
        m = pd.concat([m, summary], ignore_index=True)
        return m

    def evaluate(self):
        """
        Train on the last fold's train set and evaluate on TRAIN and TEST.

        Returns:
            pandas.DataFrame with 2 rows: Train, Test.
        """
        train_df, test_df = self.folds[-1]
        self.test_model = self.estimator.fit(train_df)

        # Evaluate on training set
        train_preds = self.test_model.transform(train_df)
        train_metric = self.evaluator.evaluate(train_preds)

        # Evaluate on test set
        test_preds = self.test_model.transform(test_df)
        test_metric = self.evaluator.evaluate(test_preds)

        self.test_metric = test_metric

        rows = [
            {"Split": "Train", **train_metric},
            {"Split": "Test", **test_metric},
        ]
        return pd.DataFrame(rows)


In [0]:
if __name__ == "__main__":
    loader = FlightDelayDataLoader()
    loader.load()

    est = MultiHeadDelayEstimator(label_col="DEP_DELAY")

    cv_set = MultiHeadCVBonus(
        estimator=est,
        version="60M",  # or "3M"/"12M" for smaller experiments
        dataloader=loader,
    )

    cv_metrics = cv_set.fit()
    print("Cross-validation metrics (Train / Val per fold + mean/std):")
    print(cv_metrics)

    test_metrics = cv_set.evaluate()
    print("Held-out TRAIN / TEST metrics:")
    print(test_metrics)